# Import Libraries

In [ ]:
import os
import time
import pickle
import random
import copy

# Analisi dati e Visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Preprocessing e Dimensionality Reduction
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# Scikit-learn: Model Selection e Pipeline
from sklearn.model_selection import (
    train_test_split,
    ParameterGrid
)
from sklearn.pipeline import Pipeline

from pytorch_tabnet.pretraining import TabNetPretrainer
from pytorch_tabnet.tab_model import TabNetClassifier

# Metriche di Valutazione
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    PrecisionRecallDisplay
)

# Deep Learning (PyTorch) e Monitoraggio
import torch
from torch import nn

import datetime

# Our packages
from utils import *
from plot import *

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn import set_config

set_config(transform_output="pandas")

KeyboardInterrupt: 

# Global Variables

In [ ]:
SEED = 42
FILENAME = "../../data/train.csv"

# Cerca la GPU
if torch.backends.mps.is_available():
    print("MPS device is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.")
    device = torch.device("cpu")

In [ ]:
def fix_random(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(SEED)

# Load the dataset

In [ ]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Preprocessing

## 1. Remove duplicates rows and columns

In [ ]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df2 = df.T.drop_duplicates().T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


##################################################
print("Nuovo # Righe: " + str(rows)+ " Nuovo # Colonne: "+str(cols) + "\n")


## 2. Label extraction and train-test splitting

In [ ]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

## 3. Pipeline

In [ ]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    """ Drop generic columns """
    def __init__(self, columns=[]):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.columns if col in X.columns])
    

class HighNanDropper(BaseEstimator, TransformerMixin):
    """ Rimuove colonne con alto numero di NaN"""
    
    def __init__(self, threshold=90):
        self.threshold = threshold
        self.columns = []

    def fit(self, X, y=None):
        nan_ratio = X.isna().mean() * 100
        self.columns = nan_ratio[nan_ratio > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.columns if col in X.columns])
    

class HighlyCorrelatedDropper(BaseEstimator, TransformerMixin):
    """" Rimozione delle feature ridondanti identificate (High Correlation > threshold) """
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.columns = []

    def fit(self, X, y=None):
        numeric_df = X.select_dtypes(include=[np.number])
        corr_matrix = numeric_df.corr().abs()
        # Selezioniamo il triangolo superiore della matrice; k=1 esclude la diagonale principale
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.columns = [column for column in upper_tri.columns if any(upper_tri[column] > self.threshold)]
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.columns if col in X.columns])
    

class NumericExtractor(BaseEstimator, TransformerMixin):
    """ Estrae feature numeriche da stringhe (36/60 mesi, polishing anni di carriera...) """
    def __init__(self, columns=[]):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        valid_cols = [col for col in self.columns if col in X.columns]
        for col in valid_cols:
            X[col] = (X[col].astype(str)
                        .replace('< 1', '0') 
                        .str.extract(r"(\d+)")
                        .astype(float))
        return X
    
    
class FeatureAverager(BaseEstimator, TransformerMixin):
    """Media di n colonne e rimuove gli originali"""
    def __init__(self, columns=[], new_name='new_avg_col'):
        self.columns = columns
        self.new_name = new_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Calcoliamo la media lungo l'asse delle righe (axis=1)
        valid_cols = [col for col in self.columns if col in X.columns]
        if valid_cols:
            X[self.new_name] = X[valid_cols].mean(axis=1)
        return X.drop(columns=[col for col in self.columns if col in X.columns])

    
class DateDifferenceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, reference_col, target_cols=None):
        self.reference_col = reference_col
        self.target_cols = target_cols or []
        self.date_format = '%b-%Y'

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        # Check if reference exists
        if self.reference_col not in X.columns:
            return X # Skip if ref is missing (maybe dropped by NaN filter)

        ref_series = pd.to_datetime(X[self.reference_col], format=self.date_format, errors='coerce')
        
        # Ensure targets is a list and filter for existence
        targets = [self.target_cols] if isinstance(self.target_cols, str) else self.target_cols
        valid_targets = [col for col in targets if col in X.columns]
        
        cols_to_drop = []

        for col in valid_targets:
            target_series = pd.to_datetime(X[col], format=self.date_format, errors='coerce')
            new_col_name = f"months_since_{col}"

            X[new_col_name] = (
                (ref_series.dt.year - target_series.dt.year) * 12 +
                (ref_series.dt.month - target_series.dt.month)
            )
            X[new_col_name] = np.round(X[new_col_name]).astype('Int64')
            cols_to_drop.append(col)

        # Drop columns
        all_drops = cols_to_drop + [self.reference_col]
        # Final safety check before drop
        X = X.drop(columns=[c for c in all_drops if c in X.columns], errors='ignore')

        return X
    

class RoundToIntTransformer(BaseEstimator, TransformerMixin):
    """ Arrotonda le colonne selezionate all'intero piu vicino """
    def __init__(self, columns=[]):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in [c for c in self.columns if c in X.columns]:
                X[col] =  np.round(X[col]).astype('Int64')
        return X
    

class CategoricalImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.categorical_cols_ = None
    
    def fit(self, X, y=None):
        self.categorical_cols_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.categorical_cols_:
            if col in X.columns:
                X[col] = X[col].fillna("__NaN__")
        return X


class NumericalMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.numerical_cols_ = None
        self.imputer_ = None
    
    def fit(self, X, y=None):
        self.numerical_cols_ = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
        if self.numerical_cols_:
            self.imputer_ = SimpleImputer(strategy='median')
            self.imputer_.fit(X[self.numerical_cols_])
        return self
    
    def transform(self, X):
        X = X.copy()
        if self.numerical_cols_ and self.imputer_:
            X[self.numerical_cols_] = self.imputer_.transform(X[self.numerical_cols_])
        return X


class CategoricalLabelEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.categorical_cols_ = None
        self.label_encoders_ = {}
        self.cat_idxs_ = []
        self.cat_dims_ = []
        self.all_columns_ = None
    
    def fit(self, X, y=None):
        self.categorical_cols_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.all_columns_ = X.columns.tolist()
        
        for col in self.categorical_cols_:
            le = LabelEncoder()
            unique_vals = X[col].astype(str).unique().tolist()
            if "__UNKNOWN__" not in unique_vals:
                unique_vals.append("__UNKNOWN__")
            
            le.fit(unique_vals)
            self.label_encoders_[col] = le
        
        self.cat_idxs_ = []
        self.cat_dims_ = []
        for i, col in enumerate(self.all_columns_):
            if col in self.categorical_cols_:
                self.cat_idxs_.append(i)
                self.cat_dims_.append(len(self.label_encoders_[col].classes_))
        
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.categorical_cols_:
            if col in X.columns:
                le = self.label_encoders_[col]
                unknown_idx = np.where(le.classes_ == "__UNKNOWN__")[0][0]
                
                X[col] = X[col].astype(str).map(
                    lambda x: le.transform([x])[0] if x in le.classes_ else unknown_idx
                )
        return X
    
    def get_cat_idxs(self):
        return self.cat_idxs_
    
    def get_cat_dims(self):
        return self.cat_dims_

class ToFloat32(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X.values.astype(np.float32)


In [ ]:
loan_performance_data_leakage = [
    'loan_status_current_code',                         # prestito in regola, in ritardo, totalmente pagato...
    'outstanding_principal_balance',                    # "outstanding principal" e' la parte del capitale da restituire
    'outstanding_principal_investor_side',              # similmente
    'total_payment_received',                           # somma pagata al creditore
    'total_payment_investor_side',
    'total_received_principal',                         # somma pagata al creditore che copre la il capitale del prestito
    'total_received_interest',                          # ... copre gli interessi
    'total_received_late_fees',                         # ... copre le penali
    'recoveries_cash',                                  # somma recuperata dopo un prestito andato in default
    'collection_recovery_fee',                          # spese per il recupero crediti
    'last_payment_date',                                # data ultimo pagamento effettuato
    'last_payment',                                     # importo ultimo pagamento
    'next_payment_date',                                # data prossimo pagamento
    'last_credit_pull_date',                            # data ultimo check profilo creditizio, durante il periodo di prestito
    'last_fico_score_high_bound',                       # ultimo punteggio FICO rilevato: al momento della concessione del prestito si usa 'fico_score_low_bound', 'fico_score_high_bound
    'last_fico_score_low_bound',
    'total_collection_amount',
    'loan_payment_installments_count'                   # potrebbe semprare il numero di rate, ma la tipologia di valori contenuti fa pensare al valore economico della singola rata (calcolo derivante di interest rate)
]
# "Settlement" indica una situazione avvenuta durante / dopo il prestito, non al momento della concessione
settlement_data_leakage = [col for col in X.columns if 'settlement' in col]
# "Hardship loans" sono concessioni per agevolare il pagamento di un prestito quando il debitore si trova in momenti di difficoltà economica (perdita lavoro, problemi medici, disastri naturali)
# https://www.oaic.gov.au/privacy/your-privacy-rights/credit-reporting/hardship-assistance/what-is-a-financial-hardship-arrangement
# https://financialrights.org.au/factsheet/financial-hardship/
hardship_data_leakage = [col for col in X.columns if 'hardship' in col]
other_leakage = [
    'original_projected_additional_accrued_interest',           # interesse addizionale previsto, presumibilmente in seguito a modifiche di piani ammortamento o hardship
    #'loan_issue_date',                                         # Il grade è influenzato dalla situazione creditizia del richiedente, più che dal periodo
                                                                # droppato in un secondo momento, dopo averlo usato per feature extraction
    'investor_side_funded_amount',
    'loan_portfolio_total_funded',
]
# Il tasso di interesse di un prestito è calcolato basandosi sul Grading assegnato al prestito stesso.
# Essendo una conseguenza del nostro target "grade", è da considerarsi data leakage
# https://www.airtel.in/blog/personal-loan/how-does-loan-grading-work/ (Accessed 02/02/2026)
loan_contract_interest_rate = [
    'loan_contract_interest_rate'
]
other_non_significant = [
    'platform_policy_code_id',                                      # id interno al prestatore
    'loan_title',                                                   # non significant column, grande sparsita' di dati. Sufficiente loan_purpose_category come aggregazione di scopo del prestito
    'borrower_address_zip',                                         # non significant column, esiste una colonna per identificazione stati
]

joint_and_secondary_cols = [
    'joint_income_annual',
    'joint_dti_ratio',
    'joint_income_verification_status',
    'joint_revolving_balance',
    'secondary_applicant_fico_low',
    'secondary_applicant_fico_high',
    'secondary_applicant_earliest_credit_line',
    'secondary_applicant_inquiries_6m',
    'secondary_applicant_mortgage_accounts',
    'secondary_applicant_open_accounts',
    'secondary_applicant_revolving_utilization',
    'secondary_applicant_open_active_installment_loans',
    'secondary_applicant_revolving_accounts',
    'secondary_applicant_chargeoffs_12m',
    'secondary_applicant_collections_12m_ex_med',
    'secondary_applicant_months_since_last_major_derog',
]

number_from_string_cols = [
    'loan_contract_term_months',
    'borrower_profile_employment_length',
]

average_cols = [
    'fico_score_low_bound',
    'fico_score_high_bound'
]

date_diff_reference = 'loan_issue_date'
date_diff_target = [ 'credit_history_earliest_line' ]


# Lista delle feature che richiedono valori interi: conteggi di linee di credito e conti, mesi, fico score
round_to_nearest_int = [
    'loan_contract_term_months', 'credit_delinquencies_2yrs', 'credit_inquiries_6m',
    'months_since_last_delinquency', 'months_since_last_public_record', 
    'credit_open_accounts', 'credit_public_records', 'credit_total_accounts',
    'collections_12m_ex_med', 'months_since_last_major_derog', 'accounts_now_delinquent',
    'open_accounts_6m', 'open_active_installment_loans', 'open_installment_loans_12m',
    'open_installment_loans_24m', 'months_since_recent_installment_loan', 
    'open_revolving_accounts_12m', 'open_revolving_accounts_24m', 'finance_inquiries',
    'credit_union_trades_total', 'credit_inquiries_12m', 'accounts_open_past_24m',
    'chargeoffs_within_12m', 'months_since_oldest_installment_acct', 
    'months_since_oldest_revolving_acct', 'months_since_recent_revolving_acct',
    'months_since_recent_trade_line', 'mortgage_accounts', 'months_since_recent_bankcard',
    'months_since_recent_bankcard_delinquency', 'months_spen_installment_loans_12m',
    'open_installment_loans_24m', 'months_since_recent_installment_loan', 
    'open_revolving_accounts_12m', 'open_revolving_accounts_24m', 'finance_inquiries',
    'credit_union_trades_total', 'credit_inquiries_12m', 'accounts_open_past_24m',
    'chargeoffs_within_12m', 'months_since_oldest_installment_acct', 
    'months_since_oldest_revolving_acct', 'months_since_recent_revolving_acct',
    'months_since_recent_trade_line', 'mortgage_accounts', 'months_since_recent_bankcard',
    'months_since_recent_bankcard_delinquency', 'months_since_recent_inquiry',
    'months_since_recent_revolving_delinquency', 'accounts_ever_120dpd',
    'active_bankcard_tradelines', 'active_revolving_tradelines', 
    'bankcard_satisfactory_accounts', 'bankcard_tradelines', 'installment_tradelines',
    'open_revolving_tradelines', 'revolving_accounts', 'tradelines_120dpd_2m',
    'tradelines_30dpd', 'tradelines_90dpd_24m', 'tradelines_open_past_12m',
    'public_record_bankruptcies', 'tax_liens_total', 'fico_average', 
    'months_since_credit_history_earliest_line' # newly created
]

categorical_to_unknown_cols = [
  'borrower_address_state',
  'loan_purpose_category',
  'borrower_income_verification_status',
  'borrower_housing_ownership_status'
]

fill_big_cols = [
    'months_since_last_public_record',
    'months_since_recent_bankcard_delinquency',
    'months_since_last_major_derog',
    'months_since_recent_revolving_delinquency',
    'months_since_last_delinquency', 
    'months_since_recent_inquiry', 
    'months_since_recent_bankcard',
    'months_since_recent_trade_line'
]

fill_zero_cols = [
    'open_accounts_6m', 'open_installment_loans_12m', 'open_installment_loans_24m',
    'open_revolving_accounts_12m', 'open_revolving_accounts_24m', 'finance_inquiries',
    'credit_inquiries_12m', 'delinquency_amount', 
    'tax_liens_total', 'mortgage_accounts', 'chargeoffs_within_12m', 
    'collections_12m_ex_med', 'accounts_now_delinquent', 'public_record_bankruptcies',
    'credit_public_records', 'credit_delinquencies_2yrs',
    'borrower_profile_employment_length',

    'tradelines_120dpd_2m', 'tradelines_30dpd', 'tradelines_90dpd_24m', 
    'accounts_ever_120dpd', 'credit_union_trades_total', 'open_active_installment_loans', 
    'credit_inquiries_6m', 'tradelines_open_past_12m', 'accounts_open_past_24m',
    'open_revolving_tradelines', 'active_bankcard_tradelines', 'installment_tradelines',
    'revolving_accounts', 'bankcard_satisfactory_accounts',
    'active_revolving_tradelines',  'bankcard_tradelines',
    'credit_open_accounts', 'credit_total_accounts'
]

fill_to_mode_cat = [
    'disbursement_method_type',
    'application_type_label',
    'listing_initial_status',
    'loan_payment_plan_flag'         # n/y
]

fill_to_mode_num = [
    'loan_contract_term_months'    # 36/60 mesi
]

In [ ]:

class CompletePipeline:
    
    def __init__(self, structure_pipeline, test_size=0.25, random_state=42):
        self.structure_pipeline = structure_pipeline
        self.test_size = test_size
        self.random_state = random_state
        
        # Initialize preprocessing steps
        self.cat_imputer = CategoricalImputer()
        self.num_imputer = NumericalMedianImputer()
        self.label_encoder = CategoricalLabelEncoder()
        self.to_float = ToFloat32()
        
        self.is_fitted_ = False
    
    def fit(self, X, y=None):
        X_transformed = self.structure_pipeline.fit_transform(X)
        
        self.cat_imputer.fit(X_transformed)
        X_transformed = self.cat_imputer.transform(X_transformed)
        
        self.num_imputer.fit(X_transformed)
        X_transformed = self.num_imputer.transform(X_transformed)
        
        self.label_encoder.fit(X_transformed)
        
        self.is_fitted_ = True
        return self
    
    def transform(self, X):
        if not self.is_fitted_:
            raise ValueError("La Pipeline deve essere fittata prima del transform.")
        
        # Apply all transformations
        X_transformed = self.structure_pipeline.transform(X)
        X_transformed = self.cat_imputer.transform(X_transformed)
        X_transformed = self.num_imputer.transform(X_transformed)
        X_transformed = self.label_encoder.transform(X_transformed)
        X_transformed = self.to_float.transform(X_transformed)
        
        return X_transformed
    
    def fit_transform(self, X, y=None):
        self.fit(X, y)
        return self.transform(X)
    
    def get_cat_idxs(self):
        if not self.is_fitted_:
            raise ValueError("La Pipeline deve essere ancora fittata.")
        return self.label_encoder.get_cat_idxs()
    
    def get_cat_dims(self):
        if not self.is_fitted_:
            raise ValueError("La Pipeline deve essere ancora fittata.")
        return self.label_encoder.get_cat_dims()
    
    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
        print(f"Pipeline salvata in {filepath}")
    
    @staticmethod
    def load(filepath):
        with open(filepath, 'rb') as f:
            pipeline = pickle.load(f)
        print(f"Pipeline caricata da {filepath}")
        return pipeline


In [ ]:

def create_train_val_split(X, y, pipeline, test_size=0.25, random_state=42):
    X_tabular = pipeline.structure_pipeline.fit_transform(X)
    
    X_train_tab, X_val_tab, Y_train, Y_val = train_test_split(
        X_tabular, y, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=y
    )
    
    pipeline.cat_imputer.fit(X_train_tab)
    X_train_tab = pipeline.cat_imputer.transform(X_train_tab)
    X_val_tab = pipeline.cat_imputer.transform(X_val_tab)
    
    pipeline.num_imputer.fit(X_train_tab)
    X_train_tab = pipeline.num_imputer.transform(X_train_tab)
    X_val_tab = pipeline.num_imputer.transform(X_val_tab)
    
    pipeline.label_encoder.fit(X_train_tab)
    X_train_tab = pipeline.label_encoder.transform(X_train_tab)
    X_val_tab = pipeline.label_encoder.transform(X_val_tab)
    
    # Convert to numpy
    X_train_np = pipeline.to_float.transform(X_train_tab)
    X_val_np = pipeline.to_float.transform(X_val_tab)
    Y_train_np = Y_train.values if hasattr(Y_train, 'values') else Y_train
    Y_val_np = Y_val.values if hasattr(Y_val, 'values') else Y_val
    
    pipeline.is_fitted_ = True
    
    return {
        'X_train_np': X_train_np,
        'X_val_np': X_val_np,
        'Y_train_np': Y_train_np,
        'Y_val_np': Y_val_np,
        'cat_idxs': pipeline.get_cat_idxs(),
        'cat_dims': pipeline.get_cat_dims()
    }



In [ ]:
def run_tabnet_grid_search(
    param_grid,
    X_train,
    Y_train,
    X_val,
    Y_val,
    cat_idxs,
    cat_dims,
    save_path="tabnet",
    load_best=False,
):
    best_loss = float("inf")
    best_balanced_acc = 0.0
    best_model = None
    best_params = None
    best_time = 0

    grid = list(ParameterGrid(param_grid))
    n_comb = len(grid)

    print("="*70)
    print(f"INIZIO GRID SEARCH TABNET")
    print(f"Totale combinazioni da testare: {n_comb}")
    print(f"Campioni di addestramento: {X_train.shape[0]}, Feature: {X_train.shape[1]}")
    print(f"Feature categoriche: {len(cat_idxs)}")
    print("="*70 + "\n")

    for i, params in enumerate(grid):
        print("\n" + "="*70)
        print(f"[ESECUZIONE {i+1}/{n_comb}] INIZIO")
        print(f"Configurazione: {params}")
        print("="*70)

        start_time = time.time()

        # Estrazione parametri con default
        n_d = params.get('n_d', 64)
        n_a = params.get('n_a', 64)
        n_steps = params.get('n_steps', 5)
        gamma = params.get('gamma', 1.5)
        n_independent = params.get('n_independent', 2)
        n_shared = params.get('n_shared', 2)
        epsilon = params.get('epsilon', 1e-15)
        lr = params.get('lr', 0.02)
        momentum = params.get('momentum', 0.02)
        pretraining_ratio = params.get('pretraining_ratio', 0.5)
        batch_size = params.get('batch_size', 1024)
        patience = params.get('patience', 10)
        max_epochs = params.get('epochs', 100)

        # Pre-addestramento
        print(f"\n>>> Fase 1/2: Pre-addestramento non supervisionato (max {max_epochs} epoche, pazienza={patience})...")
        pretrain_start = time.time()
        
        unsupervised_model = TabNetPretrainer(
            n_d=n_d, n_a=n_d, n_steps=n_steps, gamma=gamma,
            n_independent=n_independent, n_shared=n_shared,
            cat_idxs=cat_idxs, cat_dims=cat_dims,
            lambda_sparse=1e-3,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=lr),
            mask_type="sparsemax",
            device_name=device.type,
            verbose=1
        )

        unsupervised_model.fit(
            X_train=X_train,
            eval_set=[X_val],
            max_epochs=max_epochs,
            patience=patience,
            batch_size=batch_size,
            virtual_batch_size=128,
            num_workers=0,
            drop_last=False,
            pretraining_ratio=pretraining_ratio
        )
        
        pretrain_time = time.time() - pretrain_start
        print(f">>> Pre-addestramento completato in {pretrain_time:.2f}s ({pretrain_time/60:.2f}min)")

        # Addestramento
        print(f"\n>>> Fase 2/2: Addestramento supervisionato (max {max_epochs} epoche, pazienza={patience})...")
        train_start = time.time()
        
        model = TabNetClassifier(
            n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
            n_independent=n_independent, n_shared=n_shared,
            momentum=momentum,
            cat_idxs=cat_idxs, cat_dims=cat_dims,
            lambda_sparse=1e-3,
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=lr),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params={"step_size":10, "gamma":0.9},
            mask_type="sparsemax",
            device_name=device.type,
            verbose=1
        )

        model.fit(
            X_train=X_train, y_train=Y_train,
            eval_set=[(X_train, Y_train), (X_val, Y_val)],
            eval_metric=["logloss", "accuracy", "balanced_accuracy"],
            patience=patience,
            batch_size=batch_size,
            virtual_batch_size=128,
            num_workers=0,
            drop_last=True,
            max_epochs=max_epochs,
            from_unsupervised=unsupervised_model
        )
        
        train_time = time.time() - train_start
        total_time = time.time() - start_time
        
        print(f">>> Addestramento supervisionato completato in {train_time:.2f}s ({train_time/60:.2f}min)")
        
        # Estrazione metriche
        val_loss = min(model.history["val_1_logloss"])
        val_acc = max(model.history["val_1_accuracy"])
        
        # Calcolo balanced accuracy
        y_pred = model.predict(X_val)
        balanced_acc = balanced_accuracy_score(Y_val, y_pred)

        print("\n" + "-"*70)
        print(f"RISULTATI PER L'ESECUZIONE {i+1}/{n_comb}")
        print("-"*70)
        print(f"Tempo Totale:         {total_time:.2f}s ({total_time/60:.2f}min)")
        print(f"  - Pre-addestramento: {pretrain_time:.2f}s")
        print(f"  - Addestramento:     {train_time:.2f}s")
        print(f"Val Loss:             {val_loss:.64f}")
        print(f"Val Accuracy:         {val_acc:.4f}")
        print(f"Balanced Accuracy:    {balanced_acc:.4f}")
        print("-"*70)

        is_best = False
        if val_loss < best_loss:
            is_best = True
            best_loss = val_loss
            best_balanced_acc = balanced_acc
            best_model = copy.deepcopy(model)
            best_params = params
            best_time = total_time

            best_model.save_model(save_path)
            torch.save({
                "training_time": best_time,
                "val_loss": best_loss,
                "balanced_accuracy": best_balanced_acc,
                "hyperparameters": best_params
            }, save_path + "_meta.pth")

            print(f"\n TROVATO NUOVO MIGLIOR MODELLO!")
            print(f"   Miglior loss precedente: {best_loss:.6f} → Nuova migliore: {val_loss:.6f}")
            print(f"   Balanced Accuracy: {balanced_acc:.4f}")
            print(f"   Salvato in: {save_path}.zip")
        else:
            print(f"\n   Non migliore dell'attuale (loss: {best_loss:.6f})")

        # Riassunto a fine iterazione
        print("\n" + "="*70)
        print(f"[ESECUZIONE {i+1}/{n_comb}] COMPLETATA")
        print(f"Stato: {'✓ NUOVO MIGLIORE' if is_best else '✗ Nessun miglioramento'}")
        print(f"Miglior Val Loss attuale: {best_loss:.6f}")
        print(f"Miglior Balanced Acc attuale: {best_balanced_acc:.4f}")
        remaining = n_comb - (i + 1)
        est_time_remaining = remaining * total_time
        print(f"Esecuzioni rimanenti: {remaining}")
        print(f"Tempo stimato rimanente: {est_time_remaining/60:.1f} minuti ({est_time_remaining/3600:.1f} ore)")
        print("="*70 + "\n")

    print("\n" + "="*70)
    print("GRID SEARCH TABNET COMPLETATA")
    print("="*70)
    print(f"Miglior Val Loss:          {best_loss:.6f}")
    print(f"Miglior Balanced Accuracy: {best_balanced_acc:.4f}")
    print(f"Miglior Configurazione:    {best_params}")
    print(f"Miglior Tempo Addestramento: {best_time:.2f}s ({best_time/60:.2f}min)")
    print(f"Modello salvato in:        {save_path}.zip")
    print("="*70)

    return {
        "best_model": best_model,
        "best_params": best_params,
        "best_loss": best_loss,
        "best_balanced_acc": best_balanced_acc,
        "best_time": best_time,
    }

In [ ]:
structure_pipeline = Pipeline([
    ('drop_leakage', ColumnDropper(columns=loan_performance_data_leakage + settlement_data_leakage + hardship_data_leakage + other_leakage + loan_contract_interest_rate)),
    ('drop_non_significant', ColumnDropper(columns=other_non_significant)),
    ('drop_joint', ColumnDropper(columns=joint_and_secondary_cols)),
    ('drop_high_nan', HighNanDropper(threshold=0.9)),
    ('high_correlation', HighlyCorrelatedDropper(threshold=0.95)),
    ('extract_nums', NumericExtractor(columns=number_from_string_cols)),
    ('avg_features', FeatureAverager(columns=average_cols)),
    ('date_diff', DateDifferenceTransformer(reference_col=date_diff_reference, target_cols=date_diff_target)),
    ('rounding', RoundToIntTransformer(columns=round_to_nearest_int)),
])

complete_pipeline = CompletePipeline(
    structure_pipeline=structure_pipeline,
    test_size=0.25,
    random_state=SEED
)

complete_pipeline.save('tb_preprocessor.save')


train_val_data = create_train_val_split(X, y, complete_pipeline)

X_train_np = train_val_data['X_train_np']
X_val_np = train_val_data['X_val_np']
Y_train_np = train_val_data['Y_train_np']
Y_val_np = train_val_data['Y_val_np']
cat_idxs = train_val_data['cat_idxs']
cat_dims = train_val_data['cat_dims']

param_grid = {
    'batch_size': [128], 
    'epochs': [50], 
    'epsilon': [1e-15], 
    'lr': [0.02], 
    'momentum': [0.95], 
    'n_a': [32], 
    'n_d': [32], 
    'n_independent': [2], 
    'n_shared': [2], 
    'n_steps': [5], 
    'patience': [15]
}

results = run_tabnet_grid_search(
    param_grid=param_grid,
    X_train=X_train_np,
    Y_train=Y_train_np,
    X_val=X_val_np,
    Y_val=Y_val_np,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    save_path="tabnet_best_model"
)

best_model = results['best_model']
y_pred = best_model.predict(X_val_np)

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(Y_val_np, y_pred))

cm = confusion_matrix(Y_val_np, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - TabNet')
plt.show()

# Save the Structural Pipeline
with open("tabnet_structural_pipeline.pkl", "wb") as f:
    pickle.dump(structure_pipeline, f)
print("Pipeline saved.")